# 📈 Industry-Level Machine Learning Project — Stock Market Direction Prediction

### Goal
Predict whether the **next trading day's Close will be higher than today's Close**.

**Target**
- `1` = next day UP
- `0` = next day NOT UP

### Industry workflow
Business problem → data acquisition → EDA → feature engineering → leakage-safe target → chronological split → baseline → multiple ML algorithms → comparison → threshold analysis → feature importance → new-data inference → model persistence → monitoring ideas.

> **Educational project only.** Stock prices are noisy and non-stationary. Model output is not financial advice and is not a guaranteed trading signal.

In [ ]:
# Cell 1 — Install libraries in Google Colab
!pip -q install yfinance xgboost

In [ ]:
# Cell 2 — Imports and reproducibility
import warnings, json, joblib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import yfinance as yf

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, roc_curve, precision_recall_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    ExtraTreesClassifier, HistGradientBoostingClassifier
)
from sklearn.svm import SVC
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

np.random.seed(42)
pd.set_option("display.max_columns", 100)
print("Libraries ready.")

## 1. Data acquisition

Default example uses **NIFTY 50 index (`^NSEI`)** via Yahoo Finance. You can change `TICKER` to another supported symbol.

We request a long history because rolling indicators such as SMA-200 need enough observations.

In [ ]:
# Cell 3 — Download historical OHLCV data
TICKER = "^NSEI"
START_DATE = "2010-01-01"

df = yf.download(
    TICKER,
    start=START_DATE,
    auto_adjust=False,
    progress=False
)

# yfinance can sometimes return MultiIndex columns.
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df = df.reset_index()

required = ["Date", "Open", "High", "Low", "Close", "Volume"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[required].copy()
print("Shape:", df.shape)
display(df.head())
display(df.tail())

In [ ]:
# Cell 4 — Basic data audit
print(df.info())
print("\nMissing values:")
display(df.isna().sum().to_frame("missing"))

print("\nDuplicate dates:", df["Date"].duplicated().sum())
print("\nNumeric summary:")
display(df.describe().T)

## 2. Exploratory Data Analysis

Before modeling, understand price trend, returns, volume, volatility, distributions, correlations and market regimes.  
Charts are **diagnostic tools**, not proof that a pattern will continue.

In [ ]:
# Cell 5 — Closing price trend
plt.figure(figsize=(14,5))
plt.plot(df["Date"], df["Close"])
plt.title(f"{TICKER} — Closing Price")
plt.xlabel("Date")
plt.ylabel("Close")
plt.grid(alpha=.25)
plt.show()

In [ ]:
# Cell 6 — Daily returns distribution
df["Return_1D"] = df["Close"].pct_change()

plt.figure(figsize=(10,5))
plt.hist(df["Return_1D"].dropna(), bins=80)
plt.title("Distribution of Daily Returns")
plt.xlabel("Daily return")
plt.ylabel("Frequency")
plt.grid(alpha=.2)
plt.show()

print(df["Return_1D"].describe())

In [ ]:
# Cell 7 — Rolling volatility
df["Volatility_20"] = df["Return_1D"].rolling(20).std() * np.sqrt(252)

plt.figure(figsize=(14,5))
plt.plot(df["Date"], df["Volatility_20"])
plt.title("20-Day Annualized Rolling Volatility")
plt.xlabel("Date")
plt.ylabel("Volatility")
plt.grid(alpha=.25)
plt.show()

In [ ]:
# Cell 8 — Volume trend
plt.figure(figsize=(14,5))
plt.plot(df["Date"], df["Volume"])
plt.title("Trading Volume")
plt.xlabel("Date")
plt.ylabel("Volume")
plt.grid(alpha=.25)
plt.show()

## 3. Feature engineering

We create features using **only information available at or before day t**.

Features include:
- returns and momentum
- moving averages / EMA
- RSI
- MACD
- Bollinger-band position
- ATR-like range
- rolling volatility
- volume ratios
- candle/body/range features
- calendar features

The target uses `Close.shift(-1)` and is created separately.

In [ ]:
# Cell 9 — Technical feature functions
def compute_rsi(close, period=14):
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/period, adjust=False, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def engineer_features(data):
    x = data.copy().sort_values("Date").reset_index(drop=True)

    # Returns / momentum
    x["Return_1D"] = x["Close"].pct_change(1)
    x["Return_2D"] = x["Close"].pct_change(2)
    x["Return_5D"] = x["Close"].pct_change(5)
    x["Return_10D"] = x["Close"].pct_change(10)
    x["Return_20D"] = x["Close"].pct_change(20)

    # Moving averages
    for w in [5, 10, 20, 50, 100, 200]:
        x[f"SMA_{w}"] = x["Close"].rolling(w).mean()
        x[f"Close_to_SMA_{w}"] = x["Close"] / x[f"SMA_{w}"] - 1

    x["EMA_12"] = x["Close"].ewm(span=12, adjust=False).mean()
    x["EMA_26"] = x["Close"].ewm(span=26, adjust=False).mean()
    x["MACD"] = x["EMA_12"] - x["EMA_26"]
    x["MACD_Signal"] = x["MACD"].ewm(span=9, adjust=False).mean()
    x["MACD_Hist"] = x["MACD"] - x["MACD_Signal"]

    # RSI
    x["RSI_14"] = compute_rsi(x["Close"], 14)

    # Bollinger features
    x["BB_Mid"] = x["Close"].rolling(20).mean()
    x["BB_Std"] = x["Close"].rolling(20).std()
    x["BB_Upper"] = x["BB_Mid"] + 2 * x["BB_Std"]
    x["BB_Lower"] = x["BB_Mid"] - 2 * x["BB_Std"]
    x["BB_Position"] = (x["Close"] - x["BB_Lower"]) / (x["BB_Upper"] - x["BB_Lower"])

    # Volatility
    for w in [5, 10, 20, 60]:
        x[f"Volatility_{w}"] = x["Return_1D"].rolling(w).std()

    # Candle / range features
    prev_close = x["Close"].shift(1)
    tr = pd.concat([
        x["High"] - x["Low"],
        (x["High"] - prev_close).abs(),
        (x["Low"] - prev_close).abs()
    ], axis=1).max(axis=1)

    x["ATR_14"] = tr.rolling(14).mean()
    x["ATR_Pct"] = x["ATR_14"] / x["Close"]
    x["Daily_Range_Pct"] = (x["High"] - x["Low"]) / x["Open"]
    x["Body_Pct"] = (x["Close"] - x["Open"]) / x["Open"]
    x["Gap_Pct"] = (x["Open"] - prev_close) / prev_close

    # Volume
    x["Volume_MA_20"] = x["Volume"].rolling(20).mean()
    x["Volume_Ratio_20"] = x["Volume"] / x["Volume_MA_20"].replace(0, np.nan)
    x["Volume_Change"] = x["Volume"].pct_change()

    # Calendar
    x["DayOfWeek"] = x["Date"].dt.dayofweek
    x["Month"] = x["Date"].dt.month

    return x

feat = engineer_features(df)
display(feat.tail())

In [ ]:
# Cell 10 — Visualize moving averages
plt.figure(figsize=(14,6))
plt.plot(feat["Date"], feat["Close"], label="Close")
plt.plot(feat["Date"], feat["SMA_20"], label="SMA 20")
plt.plot(feat["Date"], feat["SMA_50"], label="SMA 50")
plt.plot(feat["Date"], feat["SMA_200"], label="SMA 200")
plt.title("Price and Moving Averages")
plt.legend()
plt.grid(alpha=.25)
plt.show()

In [ ]:
# Cell 11 — RSI graph
plt.figure(figsize=(14,4))
plt.plot(feat["Date"], feat["RSI_14"])
plt.axhline(70, linestyle="--")
plt.axhline(30, linestyle="--")
plt.title("RSI (14)")
plt.ylabel("RSI")
plt.grid(alpha=.25)
plt.show()

In [ ]:
# Cell 12 — MACD graph
plt.figure(figsize=(14,5))
plt.plot(feat["Date"], feat["MACD"], label="MACD")
plt.plot(feat["Date"], feat["MACD_Signal"], label="Signal")
plt.axhline(0, linewidth=1)
plt.title("MACD")
plt.legend()
plt.grid(alpha=.25)
plt.show()

In [ ]:
# Cell 13 — Bollinger Bands
recent = feat.tail(500)

plt.figure(figsize=(14,6))
plt.plot(recent["Date"], recent["Close"], label="Close")
plt.plot(recent["Date"], recent["BB_Mid"], label="20D Mean")
plt.plot(recent["Date"], recent["BB_Upper"], label="Upper Band")
plt.plot(recent["Date"], recent["BB_Lower"], label="Lower Band")
plt.title("Bollinger Bands — Recent 500 Trading Days")
plt.legend()
plt.grid(alpha=.25)
plt.show()

## 4. Target creation — avoid leakage

For row **t**, target asks whether **Close(t+1) > Close(t)**.

The final historical row has no known future close, so it must **not** be included in supervised training.

In [ ]:
# Cell 14 — Build target
feat["Next_Close"] = feat["Close"].shift(-1)
feat["Target"] = np.where(
    feat["Next_Close"].notna(),
    (feat["Next_Close"] > feat["Close"]).astype(int),
    np.nan
)

feature_cols = [
    "Return_1D","Return_2D","Return_5D","Return_10D","Return_20D",
    "Close_to_SMA_5","Close_to_SMA_10","Close_to_SMA_20",
    "Close_to_SMA_50","Close_to_SMA_100","Close_to_SMA_200",
    "MACD","MACD_Signal","MACD_Hist","RSI_14","BB_Position",
    "Volatility_5","Volatility_10","Volatility_20","Volatility_60",
    "ATR_Pct","Daily_Range_Pct","Body_Pct","Gap_Pct",
    "Volume_Ratio_20","Volume_Change","DayOfWeek","Month"
]

model_df = feat.dropna(subset=feature_cols + ["Target"]).copy()
model_df["Target"] = model_df["Target"].astype(int)

print("Model rows:", len(model_df))
print("Date range:", model_df["Date"].min(), "to", model_df["Date"].max())
display(model_df[["Date","Close","Next_Close","Target"] + feature_cols[:5]].tail())

In [ ]:
# Cell 15 — Target balance
counts = model_df["Target"].value_counts().sort_index()
print(counts)
print("\nPercentages:")
print((counts / counts.sum() * 100).round(2))

plt.figure(figsize=(6,4))
plt.bar(["Not Up (0)", "Up (1)"], [counts.get(0,0), counts.get(1,0)])
plt.title("Target Class Distribution")
plt.ylabel("Rows")
plt.show()

In [ ]:
# Cell 16 — Correlation heatmap without seaborn
corr_cols = [
    "Target","Return_1D","Return_5D","Return_20D","RSI_14",
    "MACD_Hist","BB_Position","Volatility_20","ATR_Pct",
    "Volume_Ratio_20","Body_Pct","Gap_Pct"
]
corr = model_df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(11,8))
im = ax.imshow(corr.values, aspect="auto")
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=75, ha="right")
ax.set_yticklabels(corr.index)
fig.colorbar(im, ax=ax)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

## 5. Chronological train / validation / test split

**Do not randomly shuffle financial time-series rows.**

We use:
- first 70% → training
- next 15% → validation
- final 15% → untouched test

Validation helps compare models. Test is used only after selecting the model.

In [ ]:
# Cell 17 — Chronological split
n = len(model_df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = model_df.iloc[:train_end].copy()
val = model_df.iloc[train_end:val_end].copy()
test = model_df.iloc[val_end:].copy()

X_train, y_train = train[feature_cols], train["Target"]
X_val, y_val = val[feature_cols], val["Target"]
X_test, y_test = test[feature_cols], test["Target"]

print("TRAIN:", train["Date"].min(), "→", train["Date"].max(), len(train))
print("VALID:", val["Date"].min(), "→", val["Date"].max(), len(val))
print("TEST :", test["Date"].min(), "→", test["Date"].max(), len(test))

In [ ]:
# Cell 18 — Visual split
plt.figure(figsize=(14,5))
plt.plot(model_df["Date"], model_df["Close"])
plt.axvline(val["Date"].iloc[0], linestyle="--", label="Validation starts")
plt.axvline(test["Date"].iloc[0], linestyle="--", label="Test starts")
plt.title("Chronological Train / Validation / Test Split")
plt.legend()
plt.grid(alpha=.25)
plt.show()

## 6. Baseline

A model must beat a simple reference meaningfully.  
Here the baseline predicts the majority class observed in training.

In [ ]:
# Cell 19 — Majority baseline
majority_class = int(y_train.mode().iloc[0])
baseline_pred = np.full(len(y_val), majority_class)

baseline_acc = accuracy_score(y_val, baseline_pred)
print("Training majority class:", majority_class)
print("Validation baseline accuracy:", round(baseline_acc, 4))

## 7. Train multiple algorithms

We compare linear, distance-based, kernel, bagging and boosting models.

Scaling is placed inside pipelines where required, preventing preprocessing leakage.

In [ ]:
# Cell 20 — Model definitions
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42))
    ]),
    "KNN": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=15, weights="distance"))
    ]),
    "SVM-RBF": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVC(C=1.0, kernel="rbf", probability=True, class_weight="balanced", random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=500, max_depth=8, min_samples_leaf=8,
            class_weight="balanced_subsample", n_jobs=-1, random_state=42
        ))
    ]),
    "Extra Trees": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesClassifier(
            n_estimators=500, max_depth=10, min_samples_leaf=5,
            class_weight="balanced", n_jobs=-1, random_state=42
        ))
    ]),
    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingClassifier(
            n_estimators=250, learning_rate=0.03, max_depth=3, random_state=42
        ))
    ]),
    "Hist Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingClassifier(
            max_iter=300, learning_rate=0.05, max_depth=6,
            l2_regularization=1.0, random_state=42
        ))
    ]),
    "XGBoost": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBClassifier(
            n_estimators=400, max_depth=4, learning_rate=0.03,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="logloss", random_state=42, n_jobs=-1
        ))
    ])
}

print("Models:", list(models))

In [ ]:
# Cell 21 — Evaluation helper
def evaluate_classifier(name, model, X, y):
    pred = model.predict(X)
    prob = model.predict_proba(X)[:,1]

    return {
        "Model": name,
        "Accuracy": accuracy_score(y, pred),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y, prob)
    }

validation_results = []
fitted_models = {}

for name, model in models.items():
    print("Training:", name)
    model.fit(X_train, y_train)
    fitted_models[name] = model
    validation_results.append(evaluate_classifier(name, model, X_val, y_val))

val_results = pd.DataFrame(validation_results).sort_values(
    ["ROC_AUC","F1"], ascending=False
).reset_index(drop=True)

display(val_results.style.format({
    "Accuracy":"{:.3f}", "Precision":"{:.3f}", "Recall":"{:.3f}",
    "F1":"{:.3f}", "ROC_AUC":"{:.3f}"
}))

In [ ]:
# Cell 22 — Model comparison graphs
metrics = ["Accuracy","Precision","Recall","F1","ROC_AUC"]

for metric in metrics:
    ordered = val_results.sort_values(metric)
    plt.figure(figsize=(10,5))
    plt.barh(ordered["Model"], ordered[metric])
    plt.xlim(0, 1)
    plt.title(f"Validation Model Comparison — {metric}")
    plt.xlabel(metric)
    plt.grid(axis="x", alpha=.2)
    plt.show()

In [ ]:
# Cell 23 — ROC curves on validation data
plt.figure(figsize=(9,7))

for name, model in fitted_models.items():
    prob = model.predict_proba(X_val)[:,1]
    fpr, tpr, _ = roc_curve(y_val, prob)
    auc = roc_auc_score(y_val, prob)
    plt.plot(fpr, tpr, label=f"{name} AUC={auc:.3f}")

plt.plot([0,1],[0,1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Validation ROC Curves")
plt.legend(fontsize=8)
plt.grid(alpha=.2)
plt.show()

## 8. Select best validation model and tune decision threshold

Default classification threshold is 0.50, but an industry workflow should inspect the precision/recall trade-off.

Here we choose a threshold maximizing validation F1 **for demonstration**. In a real trading system, threshold selection must include transaction costs, risk, slippage and business objectives.

In [ ]:
# Cell 24 — Best validation model
best_name = val_results.iloc[0]["Model"]
best_model = fitted_models[best_name]

print("Selected by validation ROC-AUC/F1:", best_name)

val_prob = best_model.predict_proba(X_val)[:,1]
precision, recall, thresholds = precision_recall_curve(y_val, val_prob)

# precision/recall arrays have one extra item vs thresholds
f1_values = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
best_idx = np.nanargmax(f1_values)
best_threshold = float(thresholds[best_idx])

print("Validation-selected threshold:", round(best_threshold, 3))
print("Validation F1 at threshold:", round(float(f1_values[best_idx]), 3))

In [ ]:
# Cell 25 — Threshold analysis graph
grid = np.arange(0.30, 0.71, 0.02)
rows = []

for t in grid:
    p = (val_prob >= t).astype(int)
    rows.append({
        "Threshold": t,
        "Precision": precision_score(y_val, p, zero_division=0),
        "Recall": recall_score(y_val, p, zero_division=0),
        "F1": f1_score(y_val, p, zero_division=0)
    })

threshold_df = pd.DataFrame(rows)

plt.figure(figsize=(10,5))
plt.plot(threshold_df["Threshold"], threshold_df["Precision"], label="Precision")
plt.plot(threshold_df["Threshold"], threshold_df["Recall"], label="Recall")
plt.plot(threshold_df["Threshold"], threshold_df["F1"], label="F1")
plt.axvline(best_threshold, linestyle="--", label="Selected threshold")
plt.title("Validation Threshold Analysis")
plt.xlabel("Probability threshold")
plt.ylabel("Score")
plt.legend()
plt.grid(alpha=.25)
plt.show()

## 9. Final refit and untouched test evaluation

After model + threshold selection, combine train and validation data, refit the selected algorithm, and evaluate once on the final test period.

In [ ]:
# Cell 26 — Refit best model on train + validation
trainval = pd.concat([train, val], axis=0)
X_trainval = trainval[feature_cols]
y_trainval = trainval["Target"]

final_model = clone(models[best_name])
final_model.fit(X_trainval, y_trainval)

test_prob = final_model.predict_proba(X_test)[:,1]
test_pred = (test_prob >= best_threshold).astype(int)

test_metrics = {
    "Accuracy": accuracy_score(y_test, test_pred),
    "Precision": precision_score(y_test, test_pred, zero_division=0),
    "Recall": recall_score(y_test, test_pred, zero_division=0),
    "F1": f1_score(y_test, test_pred, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test, test_prob)
}

display(pd.DataFrame([test_metrics]).style.format("{:.3f}"))
print(classification_report(y_test, test_pred, digits=3))

In [ ]:
# Cell 27 — Confusion matrix
ConfusionMatrixDisplay.from_predictions(y_test, test_pred)
plt.title(f"Test Confusion Matrix — {best_name}")
plt.show()

In [ ]:
# Cell 28 — Test ROC curve
fpr, tpr, _ = roc_curve(y_test, test_prob)
auc = roc_auc_score(y_test, test_prob)

plt.figure(figsize=(7,6))
plt.plot(fpr, tpr, label=f"AUC={auc:.3f}")
plt.plot([0,1],[0,1], linestyle="--")
plt.title("Final Test ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(alpha=.25)
plt.show()

In [ ]:
# Cell 29 — Predictions through time
prediction_frame = test[["Date","Close","Target"]].copy()
prediction_frame["Prob_Up"] = test_prob
prediction_frame["Predicted"] = test_pred

plt.figure(figsize=(14,5))
plt.plot(prediction_frame["Date"], prediction_frame["Prob_Up"])
plt.axhline(best_threshold, linestyle="--", label=f"Threshold={best_threshold:.2f}")
plt.title("Predicted Probability of Next-Day UP — Test Period")
plt.ylabel("P(UP)")
plt.legend()
plt.grid(alpha=.25)
plt.show()

display(prediction_frame.tail(20))

## 10. Feature importance

Permutation importance is model-agnostic: shuffle one feature and measure how much predictive performance deteriorates.

This is **association with model performance**, not proof of economic causation.

In [ ]:
# Cell 30 — Permutation feature importance on test period
perm = permutation_importance(
    final_model, X_test, y_test,
    n_repeats=10, random_state=42, scoring="roc_auc", n_jobs=-1
)

importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": perm.importances_mean,
    "Std": perm.importances_std
}).sort_values("Importance", ascending=False)

display(importance.head(15))

top = importance.head(15).sort_values("Importance")
plt.figure(figsize=(10,7))
plt.barh(top["Feature"], top["Importance"])
plt.title("Top 15 Permutation Importances")
plt.xlabel("Mean decrease in ROC-AUC")
plt.grid(axis="x", alpha=.2)
plt.show()

## 11. Add new unseen data and predict

### Recommended production approach
Do **not** manually type RSI/MACD/SMA values. Add raw OHLCV rows, append them to historical data, and run the **same feature-engineering function**.

Below we simulate a new raw trading day. Replace the values with your new day's actual OHLCV.

In [ ]:
# Cell 31 — Add one new raw OHLCV row
# Replace these demo values with real NEW data.
last = df.iloc[-1]

new_raw = pd.DataFrame([{
    "Date": pd.Timestamp(last["Date"]) + pd.Timedelta(days=1),
    "Open": float(last["Close"]) * 1.002,
    "High": float(last["Close"]) * 1.012,
    "Low": float(last["Close"]) * 0.994,
    "Close": float(last["Close"]) * 1.006,
    "Volume": float(max(last["Volume"], 1)) * 1.08
}])

display(new_raw)

In [ ]:
# Cell 32 — Recompute features and predict next-day direction
history_plus_new = pd.concat([df[required], new_raw], ignore_index=True)
new_feat = engineer_features(history_plus_new)

latest_row = new_feat.iloc[[-1]].copy()

if latest_row[feature_cols].isna().any(axis=None):
    raise ValueError("New row still has missing engineered features. Provide sufficient history.")

new_prob_up = float(final_model.predict_proba(latest_row[feature_cols])[:,1][0])
new_class = int(new_prob_up >= best_threshold)

print("Date:", latest_row["Date"].iloc[0])
print("Close:", round(float(latest_row["Close"].iloc[0]), 2))
print("Probability of next-day UP:", round(new_prob_up, 4))
print("Model class:", new_class, "(1=UP, 0=NOT UP)")
print("Threshold:", round(best_threshold, 4))

### Important interpretation
A probability such as `0.58` does **not** mean the market will definitely rise. It means the fitted model assigns 58% probability under its learned historical relationships.

Before any real deployment, add:
- brokerage + taxes + slippage
- walk-forward testing
- calibration
- drift monitoring
- risk limits
- retraining policy
- data-quality checks
- model/version logging

## 12. Walk-forward validation — closer to real deployment

One fixed validation period is useful for teaching, but time-series models should also be tested across multiple future windows.

This section performs expanding-window validation: train on past → validate on the next block → move forward.

In [ ]:
# Cell 33 — Expanding walk-forward evaluation
def walk_forward_scores(model_template, data, features, target="Target",
                        initial_train_ratio=0.60, test_size=120):
    n = len(data)
    train_end = int(n * initial_train_ratio)
    rows = []
    fold = 1

    while train_end + test_size <= n:
        tr = data.iloc[:train_end]
        va = data.iloc[train_end:train_end + test_size]

        m = clone(model_template)
        m.fit(tr[features], tr[target])

        prob = m.predict_proba(va[features])[:,1]
        pred = (prob >= 0.5).astype(int)

        rows.append({
            "Fold": fold,
            "Train_End": tr["Date"].iloc[-1],
            "Valid_Start": va["Date"].iloc[0],
            "Valid_End": va["Date"].iloc[-1],
            "Accuracy": accuracy_score(va[target], pred),
            "F1": f1_score(va[target], pred, zero_division=0),
            "ROC_AUC": roc_auc_score(va[target], prob)
        })

        train_end += test_size
        fold += 1

    return pd.DataFrame(rows)

wf = walk_forward_scores(models[best_name], model_df, feature_cols)
display(wf)

if len(wf):
    plt.figure(figsize=(11,5))
    plt.plot(wf["Fold"], wf["Accuracy"], marker="o", label="Accuracy")
    plt.plot(wf["Fold"], wf["F1"], marker="o", label="F1")
    plt.plot(wf["Fold"], wf["ROC_AUC"], marker="o", label="ROC-AUC")
    plt.title("Walk-Forward Performance by Fold")
    plt.xlabel("Fold")
    plt.ylabel("Score")
    plt.ylim(0,1)
    plt.legend()
    plt.grid(alpha=.25)
    plt.show()

    print("Walk-forward means:")
    display(wf[["Accuracy","F1","ROC_AUC"]].mean().to_frame("Mean"))

## 13. Save production artifacts

Save:
- trained pipeline
- feature list
- threshold
- ticker / project metadata

Because preprocessing lives inside the pipeline, inference uses the same transformation logic.

In [ ]:
# Cell 34 — Save model and metadata
joblib.dump(final_model, "stock_direction_model.joblib")

metadata = {
    "ticker": TICKER,
    "feature_columns": feature_cols,
    "threshold": best_threshold,
    "selected_model": best_name,
    "target_definition": "1 if next trading day's Close > current Close else 0"
}

with open("model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)

print("Saved:")
print("- stock_direction_model.joblib")
print("- model_metadata.json")

# Final project understanding

### Input
Historical OHLCV market data.

### ML task
Binary classification of next-day direction.

### Key industry lessons
1. **Time order matters** — no random split.
2. **Leakage control matters more than fancy algorithms.**
3. Compare against a **baseline**.
4. Use **validation** for model/threshold choices and keep test untouched.
5. A single test period is insufficient — inspect **walk-forward stability**.
6. Feature importance does not establish causation.
7. Production inference must reuse exactly the same feature engineering.
8. Real trading evaluation requires costs, slippage, position sizing and risk controls.

### Possible extensions
- benchmark against NIFTY / sector indices
- lagged global-market variables
- VIX / volatility index
- macro features
- probability calibration
- SHAP explanations
- Optuna hyperparameter tuning
- MLflow experiment tracking
- FastAPI inference service
- Streamlit dashboard
- scheduled retraining + drift monitoring

**Educational use only — not investment advice.**